In [1]:
# %%
# Импорты и настройки

import gc
import json
from pathlib import Path

import numpy as np
import optuna
import polars as pl

from catboost import CatBoostRegressor, Pool
from scipy.special import ndtr
from sklearn.metrics import average_precision_score


HORIZON = 24.0

TO_DROP = [
    "ид_события",
    "ид_канала_данных",
    "дата",
    "время",
    "datetime",
    "hours_since_prev_target",
]

TARGET_COLS = [
    "target_lower",
    "target_upper",
]


# Чанки можно задавать отдельно
TRAIN_CHUNKS = [*range(8)]
VAL_CHUNKS = [*range(80)]
TEST_CHUNKS = [*range(20)]


# Набор фичей после feature selection
SELECTED_STEP = 28

HISTORY_PATH = Path(
    "feature_selection_shap_rfe/history.csv"
)


# Настройки тюнинга
N_TRIALS = 50
RANDOM_STATE = 42


OUTPUT_DIR = Path(
    "survival_optuna"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRIALS_PATH = OUTPUT_DIR / "trials.csv"
BEST_PARAMS_PATH = OUTPUT_DIR / "best_params.json"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model.cbm"
SCORES_PATH = OUTPUT_DIR / "scores.csv"

/home/arseniy/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %%
# Формирование датасетов

def scan_split(
    split: str,
    chunks: list[int],
) -> pl.LazyFrame:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .drop(TO_DROP)
    )


# Схему получаем без загрузки данных
schema = scan_split(
    "train",
    [TRAIN_CHUNKS[0]],
).collect_schema()


feature_cols = [
    c
    for c in schema.names()
    if c not in TARGET_COLS
]


cat_features = [
    c
    for c in feature_cols
    if schema[c] in (
        pl.String,
        pl.Categorical,
        pl.Enum,
    )
]


print(f"All features: {len(feature_cols)}")
print(f"Cat features: {len(cat_features)}")

All features: 378
Cat features: 4


In [3]:
# %%
# Восстанавливаем фичи выбранного шага

history_df = pl.read_csv(
    HISTORY_PATH
)


removed_rows = (
    history_df
    .filter(
        pl.col("step") <= SELECTED_STEP
    )
    .select("removed_features")
    .drop_nulls()
    ["removed_features"]
    .to_list()
)


removed_features = {
    feature
    for row in removed_rows
    for feature in row.split(" | ")
    if feature
}


selected_features = [
    feature
    for feature in feature_cols
    if feature not in removed_features
]


selected_cat_features = [
    feature
    for feature in cat_features
    if feature in selected_features
]


expected_n_features = (
    history_df
    .filter(
        pl.col("step") == SELECTED_STEP
    )
    ["n_features"]
    .item()
)


print(f"Step: {SELECTED_STEP}")
print(f"Selected features: {len(selected_features)}")
print(f"Expected features: {expected_n_features}")
print(f"Selected cat features: {len(selected_cat_features)}")


assert len(selected_features) == expected_n_features

Step: 28
Selected features: 98
Expected features: 98
Selected cat features: 2


In [4]:
# %%
# Создаем Pool для train и val один раз
# Target для SurvivalAft: target_lower + target_upper

def make_pool(
    split: str,
    chunks: list[int],
) -> tuple[Pool, np.ndarray, int]:

    lf = (
        scan_split(
            split,
            chunks,
        )
        .select(
            selected_features
            + TARGET_COLS
        )
    )


    if selected_cat_features:
        lf = lf.with_columns(
            pl.col(
                selected_cat_features
            ).fill_null("null")
        )


    df = lf.collect(
        engine="streaming"
    )


    y_survival = (
        df
        .select(TARGET_COLS)
        .to_numpy()
    )


    y_binary = (
        (
            (df["target_upper"] != -1)
            & (
                df["target_lower"]
                <= HORIZON
            )
        )
        .cast(pl.Int8)
        .to_numpy()
    )


    pool = Pool(
        data=df.select(
            selected_features
        ),
        label=y_survival,
        cat_features=selected_cat_features,
    )


    n_rows = df.height


    del df
    del y_survival

    gc.collect()


    return pool, y_binary, n_rows


train_pool, y_train_binary, n_train = make_pool(
    "train",
    TRAIN_CHUNKS,
)

val_pool, y_val_binary, n_val = make_pool(
    "val",
    VAL_CHUNKS,
)


print(f"Train rows: {n_train:,}")
print(f"Val rows:   {n_val:,}")

Train rows: 9,523,050
Val rows:   6,871,013


In [5]:
# %%
# Тюнинг
# Модель обучается на SurvivalAft
# Optuna максимизирует PR-AUC на validation

from catboost import CatBoostError


def aft_to_probability(
    pred: np.ndarray,
    scale: float,
) -> np.ndarray:

    return ndtr(
        (
            np.log(HORIZON)
            - pred
        )
        / scale
    )


def objective(
    trial: optuna.Trial,
) -> float:

    scale = trial.suggest_float(
        "scale",
        0.3,
        3.0,
        log=True,
    )

    params = {
        "loss_function": (
            f"SurvivalAft:"
            f"dist=Normal;"
            f"scale={scale}"
        ),
        "iterations": trial.suggest_int(
            "iterations",
            20,
            100,
        ),
        "depth": trial.suggest_int(
            "depth",
            2,
            10,
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.02,
            0.15,
            log=True,
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            30.0,
            log=True,
        ),
        "random_seed": RANDOM_STATE,
        "verbose": 0,
        "allow_writing_files": False,
    }


    try:
        model = CatBoostRegressor(
            **params
        )


        model.fit(
            train_pool,
        )


        pred_val = model.predict(
            val_pool
        )


        p_val = aft_to_probability(
            pred_val,
            scale,
        )


        score = average_precision_score(
            y_val_binary,
            p_val,
        )


    except CatBoostError as e:
        print(
            f"Trial {trial.number} failed: {e}"
        )

        raise optuna.TrialPruned()


    finally:
        if "model" in locals():
            del model

        if "pred_val" in locals():
            del pred_val

        if "p_val" in locals():
            del p_val

        gc.collect()


    return score


optuna.logging.set_verbosity(
    optuna.logging.INFO
)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
    ),
)


study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
)


print()
print(
    f"Best PR-AUC: "
    f"{study.best_value:.6f}"
)

print(
    f"Best params: "
    f"{study.best_params}"
)

[I 2026-09-21 01:47:18,773] A new study created in memory with name: no-name-5d483253-113f-4e63-b94e-288d4c6f6e03
[I 2026-09-21 01:49:42,635] Trial 0 finished with value: 0.7171264588476315 and parameters: {'scale': 0.7106591851092233, 'iterations': 97, 'depth': 8, 'learning_rate': 0.06681790353582674, 'l2_leaf_reg': 1.700037298921101}. Best is trial 0 with value: 0.7171264588476315.
[I 2026-09-21 01:50:26,041] Trial 1 pruned. 


Trial 1 failed: library/cpp/json/writer/json.cpp:259: JSON writer: invalid float value: inf


[I 2026-09-21 01:53:08,705] Trial 2 finished with value: 0.5605218851517528 and parameters: {'scale': 0.3145616317558385, 'iterations': 98, 'depth': 9, 'learning_rate': 0.030678895924589226, 'l2_leaf_reg': 1.8559980846490576}. Best is trial 0 with value: 0.7171264588476315.
[I 2026-09-21 01:54:00,690] Trial 3 finished with value: 0.1437249297543313 and parameters: {'scale': 0.4576418837415782, 'iterations': 44, 'depth': 6, 'learning_rate': 0.047753845433560704, 'l2_leaf_reg': 2.6926552514864723}. Best is trial 0 with value: 0.7171264588476315.
[I 2026-09-21 01:54:32,302] Trial 4 finished with value: 0.46700585900242053 and parameters: {'scale': 1.2273661723331353, 'iterations': 31, 'depth': 4, 'learning_rate': 0.04184263761235187, 'l2_leaf_reg': 4.7170520376251766}. Best is trial 0 with value: 0.7171264588476315.
[I 2026-09-21 01:55:16,856] Trial 5 finished with value: 0.41918791365645763 and parameters: {'scale': 1.829351732859454, 'iterations': 36, 'depth': 6, 'learning_rate': 0.0659


Best PR-AUC: 0.767664
Best params: {'scale': 0.6519151388952278, 'iterations': 100, 'depth': 9, 'learning_rate': 0.14762522256363309, 'l2_leaf_reg': 22.647264220860485}


In [8]:
# %%
# Сохраняем историю тюнинга

trials_df = pl.from_pandas(
    study.trials_dataframe()
)

trials_df.write_parquet(
    TRIALS_PATH
)


with open(
    BEST_PARAMS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        study.best_params,
        f,
        ensure_ascii=False,
        indent=2,
    )


trials_df.select(
    [
        "number",
        "value",
        "params_scale",
        "params_iterations",
        "params_depth",
        "params_learning_rate",
        "params_l2_leaf_reg",
        "state",
    ]
).sort(
    "value",
    descending=True,
).head(20)

number,value,params_scale,params_iterations,params_depth,params_learning_rate,params_l2_leaf_reg,state
i64,f64,f64,i64,i64,f64,f64,str
1,null,0.429651,24,9,0.067149,11.114989,"""PRUNED"""
32,0.767664,0.651915,100,9,0.147625,22.647264,"""COMPLETE"""
44,0.761861,0.487112,100,10,0.136082,25.352077,"""COMPLETE"""
42,0.759068,0.647747,86,10,0.138019,29.756703,"""COMPLETE"""
41,0.758352,0.648213,98,9,0.141671,24.597191,"""COMPLETE"""
…,…,…,…,…,…,…,…
28,0.737245,0.594117,89,9,0.08663,22.214043,"""COMPLETE"""
48,0.73651,0.484941,92,10,0.121447,12.470656,"""COMPLETE"""
43,0.733769,1.05436,86,10,0.133661,20.695522,"""COMPLETE"""


In [ ]:
# %%
# Обучаем лучшую модель на train

best = study.best_params.copy()

best_scale = best.pop(
    "scale"
)


best_model_params = {
    "loss_function": (
        f"SurvivalAft:"
        f"dist=Normal;"
        f"scale={best_scale}"
    ),
    **best,
    "random_seed": RANDOM_STATE,
    "verbose": 1,
    "allow_writing_files": False,
}


model = CatBoostRegressor(
    **best_model_params
)


model.fit(
    train_pool,
)


model.save_model(
    BEST_MODEL_PATH
)


print(
    f"Best scale: {best_scale:.6f}"
)

print(
    f"Best params: {best_model_params}"
)

0:	learn: 45.7089762	total: 3.22s	remaining: 5m 19s
1:	learn: 44.4966156	total: 4.96s	remaining: 4m 2s
2:	learn: 42.8164651	total: 6.83s	remaining: 3m 40s
3:	learn: 40.6270574	total: 8.61s	remaining: 3m 26s
4:	learn: 37.9157742	total: 10.8s	remaining: 3m 24s
5:	learn: 34.6973701	total: 12.6s	remaining: 3m 17s
6:	learn: 31.0077739	total: 14.7s	remaining: 3m 15s
7:	learn: 26.8939192	total: 16.3s	remaining: 3m 7s
8:	learn: 22.4095602	total: 17.9s	remaining: 3m
9:	learn: 17.6034946	total: 19.4s	remaining: 2m 54s
10:	learn: 12.5381416	total: 21.2s	remaining: 2m 51s
11:	learn: 7.2569786	total: 22.9s	remaining: 2m 48s
12:	learn: 1.8043197	total: 24.5s	remaining: 2m 44s
13:	learn: 0.5339988	total: 26.2s	remaining: 2m 41s
14:	learn: 0.5428214	total: 28.1s	remaining: 2m 39s
15:	learn: 0.5842373	total: 30.1s	remaining: 2m 37s
16:	learn: 0.6373314	total: 31.9s	remaining: 2m 35s
17:	learn: 0.6871709	total: 33.5s	remaining: 2m 32s
18:	learn: 0.7500241	total: 35.1s	remaining: 2m 29s
19:	learn: 0.8020